<a href="https://colab.research.google.com/github/Krrish9381/Hindi-Voice-Chatbot/blob/main/Hindi_voice_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update && !apt-get install -y ffmpeg
!pip install -q openai-whisper gradio openai requests soundfile

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,861 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main a

In [6]:
!pip install gtts


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.2
    Uninstalling click-8.4.2:
      Successfully uninstalled click-8.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.23.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.


In [7]:
import os
import requests
import gradio as gr
import whisper
from openai import OpenAI
from gtts import gTTS  # Fallback TTS

# ==========================================
# 1. API KEYS & CLIENT SETTINGS
# ==========================================
# Paste your actual keys here:
NVIDIA_API_KEY = "Your_NVIDIA_API_KEY"
FISH_AUDIO_API_KEY = "FISH_AUDIO_API_KEY"

# Initialize NVIDIA NIM LLM Client
nvidia_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY
)

# Load Whisper ASR model on GPU
print("Loading Whisper model...")
whisper_model = whisper.load_model("small")
print("Whisper model loaded successfully!")


# ==========================================
# 2. PIPELINE FUNCTIONS
# ==========================================

def transcribe_audio(audio_path: str) -> str:
    """Step 1: Transcribe Hindi speech using Whisper."""
    if not audio_path:
        return ""
    result = whisper_model.transcribe(audio_path, language="hi")
    return result.get("text", "").strip()


def generate_llm_response(user_text: str) -> str:
    """Step 2: Generate Hindi text response using NVIDIA LLM API."""
    if not user_text:
        return "मुझे आपकी आवाज़ सुनाई नहीं दी। कृपया फिर से बोलें।"

    system_prompt = (
        "आप एक सहायक और विनम्र AI सहायक हैं। "
        "उपयोगकर्ता के प्रश्नों का उत्तर स्पष्ट, प्रासंगिक और स्वाभाविक हिंदी में दें। "
        "अपने उत्तर संक्षिप्त और बोलचाल की भाषा में रखें।"
    )

    response = nvidia_client.chat.completions.create(
        model="meta/llama-3.3-70b-instruct",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_text}
        ],
        temperature=0.7,
        max_tokens=300
    )
    return response.choices[0].message.content.strip()


def text_to_speech_fish_audio(text: str) -> str:
    """Step 3: Convert LLM Hindi response to speech via Fish Audio API (with gTTS fallback)."""
    if not text:
        return None

    output_audio_path = "output_response.mp3"

    # Attempt Fish Audio TTS
    url = "https://api.fish.audio/v1/tts"
    headers = {
        "Authorization": f"Bearer {FISH_AUDIO_API_KEY}",
        "Content-Type": "application/json"
    }

    # Payload fixed: Do not send 'reference_id': None
    payload = {
        "text": text,
        "format": "mp3"
    }

    try:
        response = requests.post(url, json=payload, headers=headers)
        if response.status_code == 200:
            with open(output_audio_path, "wb") as f:
                f.write(response.content)
            print("[SUCCESS] Fish Audio TTS generated successfully.")
            return output_audio_path
        else:
            print(f"[FISH AUDIO ERROR] Status {response.status_code}: {response.text}")
    except Exception as e:
        print(f"[FISH AUDIO EXCEPTION] {e}")

    # Fallback to gTTS if Fish Audio API fails
    print("[FALLBACK] Using gTTS for Hindi speech output...")
    try:
        tts = gTTS(text=text, lang="hi")
        tts.save(output_audio_path)
        return output_audio_path
    except Exception as e:
        print(f"[gTTS ERROR] {e}")
        return None


def process_voice_pipeline(audio_path: str):
    """Full end-to-end voice processing pipeline."""
    if audio_path is None:
        return "कोई ऑडियो प्राप्त नहीं हुआ।", "कृपया ऑडियो रिकॉर्ड करें।", None

    user_transcript = transcribe_audio(audio_path)
    llm_output = generate_llm_response(user_transcript)
    audio_output = text_to_speech_fish_audio(llm_output)

    return user_transcript, llm_output, audio_output


# ==========================================
# 3. GRADIO USER INTERFACE
# ==========================================

with gr.Blocks(title="Hindi Voice Chatbot") as demo:
    gr.Markdown("# 🎙️ हिंदी AI वॉइस चैटबॉट (Hindi Voice Chatbot)")

    with gr.Row():
        with gr.Column(scale=1):
            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="अपनी आवाज़ रिकॉर्ड करें (Record Audio)"
            )
            submit_btn = gr.Button("उत्तर प्राप्त करें (Send)", variant="primary")

        with gr.Column(scale=1):
            user_transcript_box = gr.Textbox(
                label="1. आपका ट्रांसक्रिप्ट (User Transcript)",
                interactive=False
            )
            llm_output_box = gr.Textbox(
                label="2. AI का उत्तर (LLM Text Output)",
                interactive=False
            )
            audio_output_player = gr.Audio(
                label="3. AI वॉइस आउटपुट (LLM Audio Output)",
                autoplay=True
            )

    submit_btn.click(
        fn=process_voice_pipeline,
        inputs=[audio_input],
        outputs=[user_transcript_box, llm_output_box, audio_output_player]
    )

demo.launch(share=True, debug=True)

Loading Whisper model...
Whisper model loaded successfully!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://125712230706b544ee.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[FISH AUDIO ERROR] Status 402: {"message":"Insufficient API credit. API credit is managed independently from platform credit. Please visit https://fish.audio/app/developers to view your API credit balance or add funds.","status":402}
[FALLBACK] Using gTTS for Hindi speech output...
[FISH AUDIO ERROR] Status 402: {"message":"Insufficient API credit. API credit is managed independently from platform credit. Please visit https://fish.audio/app/developers to view your API credit balance or add funds.","status":402}
[FALLBACK] Using gTTS for Hindi speech output...
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://125712230706b544ee.gradio.live


In [8]:
!pip install -q faster-whisper gtts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 53.2 MB/s eta 0:00:00


In [20]:
import os
import requests
import gradio as gr
from faster_whisper import WhisperModel
from openai import OpenAI
from gtts import gTTS

# ==========================================
# 1. API KEYS & CLIENT SETTINGS
# ==========================================
NVIDIA_API_KEY = "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"
FISH_AUDIO_API_KEY = "b5d1af01427e41c8a6afc1c691d504b9"

# Initialize NVIDIA NIM LLM Client
nvidia_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY
)

# Load Faster-Whisper on GPU (4x faster than standard Whisper)
print("Loading Faster-Whisper model on GPU...")
whisper_model = WhisperModel("medium", device="cuda", compute_type="float16")
print("Faster-Whisper model loaded successfully!")


# ==========================================
# 2. PIPELINE FUNCTIONS
# ==========================================

def transcribe_audio(audio_path: str) -> str:
    """Step 1: Fast Hindi transcription with prompt guiding."""
    if not audio_path:
        return ""

    # Prompting forces correct Devanagari spelling for common Hindi words
    hindi_initial_prompt = "यह एक हिंदी बातचीत है। भारत, प्रधानमंत्री, नरेंद्र मोदी, नाम, क्या, बताइए, मुख्यमंत्र।"

    segments, _ = whisper_model.transcribe(
        audio_path,
        language="hi",
        initial_prompt=hindi_initial_prompt,
        beam_size=5
    )

    # Combine recognized speech segments
    transcript = " ".join([segment.text for segment in segments]).strip()
    return transcript


def generate_llm_response(user_text: str) -> str:
    """Generate expressive Hindi text using interjections and punctuation."""
    if not user_text:
        return "मुझे आपकी आवाज़ स्पष्ट सुनाई नहीं दी... कृपया फिर से बोलें।"

    system_prompt = (
        "आप एक बेहद जीवंत, संवेदी और दोस्ताना हिंदी AI सहायक हैं।\n"
        "आपको अपना उत्तर इस तरह देना है कि पढ़ते समय और सुनते समय उसमें इंसानी भावनाएं महसूस हों।\n\n"
        "दिशा-निर्देश:\n"
        "1. [bracket] जैसे टैग्स का प्रयोग बिल्कुल न करें।\n"
        "2. भावनाओं और उत्सुकता के लिए प्राकृतिक हिंदी शब्दों का प्रयोग करें जैसे: 'अरे वाह!', 'सच में?', 'ओहो...', 'अरे हाँ!'\n"
        "3. ठहराव (Pause) के लिए '...' (तीन बिंदु) और उत्साह के लिए '!' का प्रयोग करें।\n"
        "4. उत्तर को 2-3 छोटी और रोचक पंक्तियों में रखें।"
    )

    try:
        response = nvidia_client.chat.completions.create(
            model="meta/llama-3.1-8b-instruct",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_text}
            ],
            temperature=0.8,
            max_tokens=200
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"[LLM ERROR]: {e}")
        return "क्षमा करें... मुझे उत्तर तैयार करने में समस्या आ रही है।"


def text_to_speech_fish_audio(text: str) -> str:
    """Step 3: Convert LLM Hindi response to speech (Fish Audio with gTTS backup)."""
    if not text:
        return None

    output_audio_path = "output_response.mp3"

    # Attempt Fish Audio API call
    url = "https://api.fish.audio/v1/tts"
    headers = {
        "Authorization": f"Bearer {FISH_AUDIO_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "text": text,
        "format": "mp3"
    }

    try:
        response = requests.post(url, json=payload, headers=headers, timeout=5)
        if response.status_code == 200:
            with open(output_audio_path, "wb") as f:
                f.write(response.content)
            return output_audio_path
        else:
            print(f"[FISH AUDIO ERROR] Status {response.status_code}")
    except Exception as e:
        print(f"[FISH AUDIO EXCEPTION] {e}")

    # Fallback to gTTS if Fish Audio API fails/times out
    try:
        tts = gTTS(text=text, lang="hi")
        tts.save(output_audio_path)
        return output_audio_path
    except Exception as e:
        print(f"[gTTS ERROR] {e}")
        return None



import re

def process_voice_pipeline(audio_path: str):
    """Full end-to-end voice processing pipeline."""
    if audio_path is None:
        return "कोई ऑडियो प्राप्त नहीं हुआ।", "कृपया ऑडियो रिकॉर्ड करें।", None

    user_transcript = transcribe_audio(audio_path)

    # 1. LLM output with tags (for display on screen)
    llm_output = generate_llm_response(user_transcript)

    # 2. Clean text without bracketed tags (for TTS audio execution)
    # This removes anything inside [...] like [excited], [pause], [happy]
    clean_text_for_tts = re.sub(r'\[.*?\]', '', llm_output).strip()

    # 3. Replace multiple spaces/punctuation left behind
    clean_text_for_tts = re.sub(r'\s+', ' ', clean_text_for_tts)

    # Pass clean text to TTS so it doesn't read bracket words aloud
    audio_output = text_to_speech_fish_audio(clean_text_for_tts)

    return user_transcript, llm_output, audio_output


# ==========================================
# 3. GRADIO USER INTERFACE
# ==========================================

with gr.Blocks(title="Hindi Voice Chatbot") as demo:
    gr.Markdown("# 🎙️ हिंदी AI वॉइस चैटबॉट (Hindi Voice Chatbot)")

    with gr.Row():
        with gr.Column(scale=1):
            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="अपनी आवाज़ रिकॉर्ड करें (Record Audio)"
            )
            submit_btn = gr.Button("उत्तर प्राप्त करें (Send)", variant="primary")

        with gr.Column(scale=1):
            user_transcript_box = gr.Textbox(
                label="1. आपका ट्रांसक्रिप्ट (User Transcript)",
                interactive=False
            )
            llm_output_box = gr.Textbox(
                label="2. AI का उत्तर (LLM Text Output)",
                interactive=False
            )
            audio_output_player = gr.Audio(
                label="3. AI वॉइस आउटपुट (LLM Audio Output)",
                autoplay=True
            )

    submit_btn.click(
        fn=process_voice_pipeline,
        inputs=[audio_input],
        outputs=[user_transcript_box, llm_output_box, audio_output_player]
    )

demo.launch(share=True, debug=True)

Loading Faster-Whisper model on GPU...
Faster-Whisper model loaded successfully!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://290ddd6ee77c698d62.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[FISH AUDIO ERROR] Status 402
[FISH AUDIO ERROR] Status 402
[FISH AUDIO ERROR] Status 402
[FISH AUDIO ERROR] Status 402
[FISH AUDIO ERROR] Status 402
[FISH AUDIO ERROR] Status 402
[FISH AUDIO ERROR] Status 402
[FISH AUDIO ERROR] Status 402
[FISH AUDIO ERROR] Status 402
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://290ddd6ee77c698d62.gradio.live
